# Deep Learning Diagnostics Lab

이 노트북의 목적은 **작은 CNN 하나를 여러 진단 도구로 관찰하는 방법을 익히는 것**입니다.

전체 흐름은 다음과 같습니다.

```text
학습이 잘 되는가?
→ 어느 층이 실제로 움직이는가?
→ 내부 표현이 몇 개 방향을 쓰는가?
→ class 정보가 표현 안에 잘 정리되는가?
→ 표현 구조가 학습 중 언제 바뀌는가?
→ 표현 공간의 모양과 국소 구조는 어떤가?
→ parameter 공간의 loss surface는 어떤가?
```

사용 도구: loss/accuracy, gradient norm, update-to-weight ratio, effective rank, linear probe, CKA, PCA/UMAP, local PCA+kNN, Hessian top eigenvalue, weight interpolation.

**저장 정책**: CSV/NPZ/PNG/TensorBoard/JSON은 Google Drive에 저장하고, 모델 weight `.pt`는 `/content/local_checkpoints`에만 저장합니다.

## 0. 환경 설정

**이 셀의 목적**: 필요한 라이브러리를 불러오고 저장 폴더를 만듭니다.

`FAST_MODE=True`이면 Colab에서 짧게 확인할 수 있도록 CIFAR-10 일부와 6 epoch만 사용합니다.

In [ ]:
!pip -q install umap-learn tensorboard

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms

from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap

from google.colab import drive

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 7
FAST_MODE = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/deep_learning_diagnostics")
CSV_DIR = DRIVE_ROOT / "csv"
NPZ_DIR = DRIVE_ROOT / "npz"
FIG_DIR = DRIVE_ROOT / "figures"
TB_DIR = DRIVE_ROOT / "tensorboard"
SUMMARY_DIR = DRIVE_ROOT / "summaries"

# 모델 weight는 Google Drive가 아니라 Colab runtime에만 둡니다.
LOCAL_CKPT_DIR = Path("/content/local_checkpoints")

for folder in [CSV_DIR, NPZ_DIR, FIG_DIR, TB_DIR, SUMMARY_DIR, LOCAL_CKPT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

## 1. 데이터와 진단 시점 준비

**왜 학습용 입력과 진단용 입력을 나누나?**

학습 때는 crop/flip으로 이미지를 조금씩 바꾸지만, CKA나 PCA처럼 epoch 사이의 표현을 비교할 때는 **같은 이미지를 같은 순서로 넣어야** 합니다. 그래서 진단에는 augmentation이 없는 고정 입력을 씁니다.

representation은 매 epoch마다 분석하지 않고 전체 학습을 약 4구간으로 나눠 기록합니다.

- 6 epochs → `0, 2, 4, 6`
- 12 epochs → `0, 3, 6, 9, 12`

여기서 epoch 0은 **학습 전 초기 상태**입니다.

In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_aug = datasets.CIFAR10(
    root="/content/data",
    train=True,
    download=True,
    transform=train_transform,
)

train_eval = datasets.CIFAR10(
    root="/content/data",
    train=True,
    download=False,
    transform=eval_transform,
)

permutation = torch.randperm(
    len(train_aug),
    generator=torch.Generator().manual_seed(SEED),
).tolist()

if FAST_MODE:
    n_train = 12_000
    n_val = 2_000
    EPOCHS = 6
else:
    n_train = 40_000
    n_val = 5_000
    EPOCHS = 12

train_indices = permutation[:n_train]
val_indices = permutation[n_train:n_train + n_val]

train_dataset = Subset(train_aug, train_indices)
train_eval_dataset = Subset(train_eval, train_indices)
val_dataset = Subset(train_eval, val_indices)

loader_args = dict(batch_size=256, num_workers=2, pin_memory=True)

train_loader = DataLoader(train_dataset, shuffle=True, **loader_args)
train_eval_loader = DataLoader(train_eval_dataset, shuffle=False, **loader_args)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)

diag_interval = max(1, round(EPOCHS / 4))
DIAG_EPOCHS = sorted(
    set([0] + list(range(diag_interval, EPOCHS + 1, diag_interval)) + [EPOCHS])
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Diagnostic epochs:", DIAG_EPOCHS)

## 2. 진단할 CNN 정의

**이 셀의 목적**: 내부 표현을 꺼낼 위치를 명확히 만듭니다.

```text
image → stem → block1 → block2 → penultimate → head → logits
```

CNN의 중간 출력은 `(channel, height, width)` 형태이므로, 공간 평균을 내서 sample 하나당 feature vector 하나로 바꿉니다.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        self.block1 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.penultimate = nn.Linear(128, 64)
        self.head = nn.Linear(64, 10)

    def forward(self, x, return_features=False):
        features = {}

        x = self.stem(x)
        features["stem"] = x.mean(dim=(2, 3))

        x = self.block1(x)
        features["block1"] = x.mean(dim=(2, 3))

        x = self.block2(x)
        features["block2"] = x.mean(dim=(2, 3))

        x = self.pool(x).flatten(1)
        x = F.relu(self.penultimate(x))
        features["penultimate"] = x

        logits = self.head(x)

        if return_features:
            return logits, features
        return logits


LAYER_NAMES = ["stem", "block1", "block2", "penultimate"]

TRACKED_PARAMETERS = {
    "stem": "stem.0.weight",
    "block1": "block1.0.weight",
    "block2": "block2.0.weight",
    "penultimate": "penultimate.weight",
    "head": "head.weight",
}

# SGD와 AdamW를 공정하게 비교하려고 같은 초기값을 저장합니다.
torch.manual_seed(SEED)
INITIAL_STATE = {
    name: value.cpu().clone()
    for name, value in SmallCNN().state_dict().items()
}

## 3. 기본 평가 함수

**입력**: 모델 + DataLoader

**출력**: 평균 cross-entropy loss + accuracy

학습곡선, Hessian 비교, interpolation 비교에서 계속 재사용합니다.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)

        total_loss += F.cross_entropy(
            logits, y, reduction="sum"
        ).item()
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_count += y.numel()

    average_loss = total_loss / total_count
    accuracy = total_correct / total_count

    return average_loss, accuracy

## 4. 학습하면서 gradient와 실제 parameter 이동 기록

질문은 하나입니다.

> **어느 층이 실제로 얼마나 학습되고 있는가?**

매 batch에서 두 값을 기록합니다.

1. `gradient norm = ||dL/dW||` : 이 층이 받은 학습 신호의 크기
2. `update-to-weight = ||ΔW|| / ||W||` : 현재 weight 크기에 비해 실제로 얼마나 움직였는지

gradient가 커도 optimizer가 실제 parameter를 조금만 움직일 수 있으므로 둘을 같이 봅니다.

In [ ]:
def train_model(run_name, optimizer_name):
    model = SmallCNN().to(DEVICE)
    model.load_state_dict(INITIAL_STATE)

    if optimizer_name == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(), lr=0.08, momentum=0.9
        )
    elif optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=2e-3
        )
    else:
        raise ValueError(optimizer_name)

    writer = SummaryWriter(str(TB_DIR / run_name))
    history_rows = []
    dynamics_rows = []

    # epoch 0 = 학습 전 상태
    torch.save(
        model.state_dict(),
        LOCAL_CKPT_DIR / f"{run_name}_epoch0.pt",
    )

    global_step = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_count = 0

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y)

            # backward 후 gradient가 생깁니다.
            loss.backward()

            named_parameters = dict(model.named_parameters())

            # optimizer.step() 전 값을 복사해 실제 ΔW를 계산합니다.
            before_update = {
                tag: named_parameters[param_name].detach().clone()
                for tag, param_name in TRACKED_PARAMETERS.items()
            }

            dynamics_row = {
                "run": run_name,
                "epoch": epoch,
                "step": global_step,
            }

            for tag, param_name in TRACKED_PARAMETERS.items():
                parameter = named_parameters[param_name]
                grad_norm = parameter.grad.detach().norm().item()
                dynamics_row[f"{tag}_grad_norm"] = grad_norm

            # 여기서 실제 parameter update가 일어납니다.
            optimizer.step()
            named_parameters = dict(model.named_parameters())

            for tag, param_name in TRACKED_PARAMETERS.items():
                parameter = named_parameters[param_name]
                delta_norm = (
                    parameter.detach() - before_update[tag]
                ).norm().item()
                weight_norm = parameter.detach().norm().item()
                update_ratio = delta_norm / (weight_norm + 1e-12)

                dynamics_row[f"{tag}_update_to_weight"] = update_ratio

                writer.add_scalar(
                    f"gradient/{tag}",
                    dynamics_row[f"{tag}_grad_norm"],
                    global_step,
                )
                writer.add_scalar(
                    f"update_to_weight/{tag}",
                    update_ratio,
                    global_step,
                )

            dynamics_rows.append(dynamics_row)
            global_step += 1

            train_loss_sum += loss.item() * y.size(0)
            train_correct += (logits.argmax(dim=1) == y).sum().item()
            train_count += y.numel()

        val_loss, val_accuracy = evaluate(model, val_loader)

        history_rows.append({
            "run": run_name,
            "epoch": epoch,
            "train_loss": train_loss_sum / train_count,
            "train_accuracy": train_correct / train_count,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        })

        writer.add_scalar("validation/loss", val_loss, epoch)
        writer.add_scalar("validation/accuracy", val_accuracy, epoch)

        # representation을 볼 시점만 runtime에 checkpoint를 남깁니다.
        if epoch in DIAG_EPOCHS:
            torch.save(
                model.state_dict(),
                LOCAL_CKPT_DIR / f"{run_name}_epoch{epoch}.pt",
            )

        print(
            f"{run_name} epoch {epoch}: "
            f"val_loss={val_loss:.3f}, val_accuracy={val_accuracy:.3f}"
        )

    writer.close()

    history_df = pd.DataFrame(history_rows)
    dynamics_df = pd.DataFrame(dynamics_rows)

    history_df.to_csv(CSV_DIR / f"{run_name}_history.csv", index=False)
    dynamics_df.to_csv(CSV_DIR / f"{run_name}_dynamics.csv", index=False)

    return model, history_df, dynamics_df

## 5. SGD와 AdamW 실제 학습

두 모델은 **동일한 초기 weight**에서 시작하고 optimizer만 다릅니다. 이 셀이 가장 오래 걸립니다.

In [ ]:
sgd_model, sgd_history, sgd_dynamics = train_model(
    run_name="sgd",
    optimizer_name="sgd",
)

adamw_model, adamw_history, adamw_dynamics = train_model(
    run_name="adamw",
    optimizer_name="adamw",
)

## 6. 기본 학습곡선

복잡한 분석 전에 먼저 **문제가 실제로 있는지 위치를 잡습니다.**

- validation loss가 내려가는가?
- validation accuracy가 올라가는가?
- optimizer 사이 차이가 있는가?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for history, label in [
    (sgd_history, "SGD"),
    (adamw_history, "AdamW"),
]:
    axes[0].plot(
        history["epoch"],
        history["val_loss"],
        marker="o",
        label=label,
    )
    axes[1].plot(
        history["epoch"],
        history["val_accuracy"],
        marker="o",
        label=label,
    )

axes[0].set_title("Validation loss")
axes[1].set_title("Validation accuracy")

for axis in axes:
    axis.set_xlabel("Epoch")
    axis.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "training_curves.png", dpi=170)
plt.show()

## 7. Gradient norm과 update-to-weight 보기

**왼쪽**은 각 층이 받은 gradient 신호, **오른쪽**은 실제 weight가 움직인 비율입니다.

예를 들어 `gradient는 큰데 update-to-weight는 작다`면 optimizer scaling 때문에 실제 이동은 작을 수 있습니다.

In [ ]:
def plot_dynamics(dynamics_df, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for layer in TRACKED_PARAMETERS:
        gradient_curve = (
            dynamics_df[f"{layer}_grad_norm"]
            .rolling(20, min_periods=1)
            .mean()
        )
        update_curve = (
            dynamics_df[f"{layer}_update_to_weight"]
            .rolling(20, min_periods=1)
            .mean()
        )

        axes[0].plot(gradient_curve, label=layer)
        axes[1].plot(update_curve, label=layer)

    axes[0].set_title(f"{title}: gradient norm")
    axes[1].set_title(f"{title}: update / weight")

    for axis in axes:
        axis.set_yscale("log")
        axis.set_xlabel("Training step")
        axis.legend()

    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{title.lower()}_dynamics.png", dpi=170)
    plt.show()


plot_dynamics(sgd_dynamics, "SGD")
plot_dynamics(adamw_dynamics, "AdamW")

## 8. Representation snapshot 추출

각 diagnostic epoch에서 **동일한 validation 2,000장**을 넣고 네 층의 activation을 저장합니다.

```text
sample → stem feature
       → block1 feature
       → block2 feature
       → penultimate feature
```

이 NPZ가 뒤의 effective rank, CKA, PCA/UMAP, local PCA 분석의 공통 원자료입니다.

In [ ]:
@torch.no_grad()
def extract_features(model, loader, max_samples=2000):
    model.eval()

    feature_batches = {layer: [] for layer in LAYER_NAMES}
    label_batches = []
    prediction_batches = []
    collected = 0

    for x, y in loader:
        logits, features = model(
            x.to(DEVICE),
            return_features=True,
        )

        take = min(y.size(0), max_samples - collected)

        for layer in LAYER_NAMES:
            feature_batches[layer].append(
                features[layer][:take].cpu()
            )

        label_batches.append(y[:take])
        prediction_batches.append(
            logits[:take].argmax(dim=1).cpu()
        )

        collected += take
        if collected >= max_samples:
            break

    feature_arrays = {
        layer: torch.cat(parts).numpy()
        for layer, parts in feature_batches.items()
    }

    labels = torch.cat(label_batches).numpy()
    predictions = torch.cat(prediction_batches).numpy()

    return feature_arrays, labels, predictions


def load_checkpoint(run_name, epoch):
    return torch.load(
        LOCAL_CKPT_DIR / f"{run_name}_epoch{epoch}.pt",
        map_location="cpu",
    )


def make_snapshots(run_name):
    snapshots = {}

    for epoch in DIAG_EPOCHS:
        model = SmallCNN().to(DEVICE)
        model.load_state_dict(load_checkpoint(run_name, epoch))

        features, labels, predictions = extract_features(
            model,
            val_loader,
            max_samples=2000,
        )

        snapshots[epoch] = {
            "features": features,
            "labels": labels,
            "predictions": predictions,
        }

        np.savez_compressed(
            NPZ_DIR / f"{run_name}_epoch{epoch}.npz",
            labels=labels,
            predictions=predictions,
            **features,
        )

        print(f"Saved {run_name} representation at epoch {epoch}")

    return snapshots


sgd_snapshots = make_snapshots("sgd")
adamw_snapshots = make_snapshots("adamw")

## 9. Effective rank: 표현이 몇 개 방향을 사용하는가?

activation 행렬에 SVD를 적용해 표현이 실질적으로 몇 개 방향을 사용하는지 요약합니다.

- effective rank가 높음 → 여러 feature 방향 사용
- 낮음 → 소수 방향에 집중

**낮다고 바로 collapse라고 판단하지 않습니다.** 다음 linear probe와 같이 봅니다.

In [ ]:
def effective_rank(features):
    X = torch.tensor(features, dtype=torch.float32)
    X = X - X.mean(dim=0, keepdim=True)

    singular_values = torch.linalg.svdvals(X)
    spectral_power = singular_values.square()
    probability = spectral_power / spectral_power.sum().clamp_min(1e-12)

    entropy = -(
        probability * torch.log(probability.clamp_min(1e-12))
    ).sum()

    return torch.exp(entropy).item()


rank_rows = []

for run_name, snapshots in [
    ("sgd", sgd_snapshots),
    ("adamw", adamw_snapshots),
]:
    for epoch in DIAG_EPOCHS:
        for layer in LAYER_NAMES:
            rank_rows.append({
                "run": run_name,
                "epoch": epoch,
                "layer": layer,
                "effective_rank": effective_rank(
                    snapshots[epoch]["features"][layer]
                ),
            })

rank_df = pd.DataFrame(rank_rows)
rank_df.to_csv(CSV_DIR / "effective_rank_over_time.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for axis, run_name in zip(axes, ["sgd", "adamw"]):
    for layer in LAYER_NAMES:
        part = rank_df[
            (rank_df["run"] == run_name)
            & (rank_df["layer"] == layer)
        ]
        axis.plot(
            part["epoch"],
            part["effective_rank"],
            marker="o",
            label=layer,
        )

    axis.set_title(f"{run_name.upper()}: effective rank")
    axis.set_xlabel("Diagnostic epoch")
    axis.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "effective_rank_over_time.png", dpi=170)
plt.show()

## 10. Linear probe: class 정보가 표현 안에서 읽히는가?

중간 representation을 **고정**하고 그 위에 선형 분류기 하나만 새로 학습합니다.

해석 예:

```text
effective rank ↓ + probe accuracy ↑
→ class에 필요한 방향으로 압축된 가능성

effective rank ↓ + probe accuracy ↓
→ 정보 손실 후보
```

In [ ]:
train_features, train_labels, _ = extract_features(
    sgd_model,
    train_eval_loader,
    max_samples=5000,
)

final_sgd = sgd_snapshots[EPOCHS]


def linear_probe(X_train, y_train, X_val, y_val):
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train).astype("float32")
    X_val = scaler.transform(X_val).astype("float32")

    X_train = torch.tensor(X_train, device=DEVICE)
    y_train = torch.tensor(y_train, device=DEVICE)
    X_val = torch.tensor(X_val, device=DEVICE)
    y_val = torch.tensor(y_val, device=DEVICE)

    classifier = nn.Linear(X_train.shape[1], 10).to(DEVICE)
    optimizer = torch.optim.AdamW(classifier.parameters(), lr=0.08)

    for _ in range(150):
        optimizer.zero_grad(set_to_none=True)
        logits = classifier(X_train)
        loss = F.cross_entropy(logits, y_train)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        accuracy = (
            classifier(X_val).argmax(dim=1) == y_val
        ).float().mean().item()

    return accuracy


probe_rows = []

for layer in LAYER_NAMES:
    probe_rows.append({
        "layer": layer,
        "probe_accuracy": linear_probe(
            train_features[layer],
            train_labels,
            final_sgd["features"][layer],
            final_sgd["labels"],
        ),
    })

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(CSV_DIR / "linear_probe.csv", index=False)

plt.figure(figsize=(7, 4))
plt.bar(probe_df["layer"], probe_df["probe_accuracy"])
plt.ylim(0, 1)
plt.ylabel("Validation accuracy")
plt.title("Linear probe by layer")
plt.savefig(FIG_DIR / "linear_probe.png", dpi=170)
plt.show()

## 11. CKA: 표현 구조가 학습 중 언제 바뀌는가?

CKA는 두 representation이 sample들을 **비슷한 관계 구조로 배치하는지** 비교합니다.

- `CKA to init` 감소 → 초기 표현에서 멀어짐
- `CKA to final` 증가 → 최종 표현 구조에 가까워짐

따라서 어느 층이 **언제** 크게 바뀌는지 볼 수 있습니다.

In [ ]:
def linear_cka(X, Y):
    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)

    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)

    numerator = (X.T @ Y).square().sum()
    norm_x = (X.T @ X).square().sum().sqrt()
    norm_y = (Y.T @ Y).square().sum().sqrt()

    return (numerator / (norm_x * norm_y + 1e-12)).item()


cka_rows = []

for run_name, snapshots in [
    ("sgd", sgd_snapshots),
    ("adamw", adamw_snapshots),
]:
    initial_features = snapshots[0]["features"]
    final_features = snapshots[EPOCHS]["features"]

    for epoch in DIAG_EPOCHS:
        current_features = snapshots[epoch]["features"]

        for layer in LAYER_NAMES:
            cka_rows.append({
                "run": run_name,
                "epoch": epoch,
                "layer": layer,
                "cka_to_init": linear_cka(
                    current_features[layer],
                    initial_features[layer],
                ),
                "cka_to_final": linear_cka(
                    current_features[layer],
                    final_features[layer],
                ),
            })

cka_df = pd.DataFrame(cka_rows)
cka_df.to_csv(CSV_DIR / "cka_over_time.csv", index=False)

for run_name in ["sgd", "adamw"]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for layer in LAYER_NAMES:
        part = cka_df[
            (cka_df["run"] == run_name)
            & (cka_df["layer"] == layer)
        ]

        axes[0].plot(
            part["epoch"],
            part["cka_to_init"],
            marker="o",
            label=layer,
        )
        axes[1].plot(
            part["epoch"],
            part["cka_to_final"],
            marker="o",
            label=layer,
        )

    axes[0].set_title("CKA to initialization")
    axes[1].set_title("CKA to final")

    for axis in axes:
        axis.set_ylim(0, 1.02)
        axis.set_xlabel("Diagnostic epoch")
        axis.legend()

    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{run_name}_cka.png", dpi=170)
    plt.show()

## 12. PCA / UMAP: 표현 공간을 눈으로 보기

- **PCA**: 전체 point cloud에서 큰 분산 방향을 우선 보여줍니다.
- **UMAP**: 가까운 sample들의 local neighborhood 구조를 강조합니다.

검은 테두리는 그 시점에 **오분류된 sample**입니다.

질문은 `class cluster가 학습하면서 분리되는가?`, `오분류가 경계나 겹치는 영역에 몰리는가?`입니다.

In [ ]:
def plot_embedding_snapshots(run_name, snapshots):
    for epoch in DIAG_EPOCHS:
        X = snapshots[epoch]["features"]["penultimate"]
        labels = snapshots[epoch]["labels"]
        predictions = snapshots[epoch]["predictions"]

        X = StandardScaler().fit_transform(X)

        reducers = {
            "PCA": PCA(n_components=2, random_state=SEED),
            "UMAP": umap.UMAP(
                n_components=2,
                n_neighbors=20,
                min_dist=0.15,
                random_state=SEED,
            ),
        }

        for method_name, reducer in reducers.items():
            embedding = reducer.fit_transform(X)
            correct = labels == predictions

            embedding_df = pd.DataFrame({
                "x": embedding[:, 0],
                "y": embedding[:, 1],
                "label": labels,
                "prediction": predictions,
                "correct": correct,
            })

            embedding_df.to_csv(
                CSV_DIR / f"{run_name}_{method_name.lower()}_epoch{epoch}.csv",
                index=False,
            )

            plt.figure(figsize=(7, 6))
            plt.scatter(
                embedding[:, 0],
                embedding[:, 1],
                c=labels,
                cmap="tab10",
                s=10,
                alpha=0.6,
            )
            plt.scatter(
                embedding[~correct, 0],
                embedding[~correct, 1],
                facecolors="none",
                edgecolors="black",
                s=35,
            )

            plt.title(
                f"{run_name.upper()} {method_name} epoch {epoch}\n"
                "black ring = misclassified"
            )
            plt.savefig(
                FIG_DIR / f"{run_name}_{method_name.lower()}_epoch{epoch}.png",
                dpi=170,
            )
            plt.show()


plot_embedding_snapshots("sgd", sgd_snapshots)
plot_embedding_snapshots("adamw", adamw_snapshots)

## 13. Local PCA + kNN: 한 sample 주변은 얼마나 복잡한가?

각 sample에서 가장 가까운 이웃 30개를 모아 local PCA를 합니다.

`90% 분산을 설명하는 데 필요한 차원 수`를 local dimension proxy로 사용합니다.

- 작음 → 주변이 비교적 단순한 저차원 구조
- 큼 → 주변 점구름이 여러 방향으로 퍼져 있음

여기서는 정답 sample과 오답 sample 주변의 복잡도를 비교합니다.

In [ ]:
X = StandardScaler().fit_transform(
    final_sgd["features"]["penultimate"]
)

neighbor_model = NearestNeighbors(n_neighbors=31).fit(X)
_, neighbor_indices = neighbor_model.kneighbors(X)

local_dimensions = []

for sample_index in range(len(X)):
    # 첫 번째 이웃은 자기 자신이므로 제외합니다.
    local_points = X[neighbor_indices[sample_index, 1:]]
    local_points = local_points - local_points.mean(axis=0)

    singular_values = np.linalg.svd(
        local_points,
        compute_uv=False,
    )
    local_power = singular_values ** 2
    cumulative_ratio = np.cumsum(local_power) / local_power.sum()

    local_dimension = np.searchsorted(cumulative_ratio, 0.90) + 1
    local_dimensions.append(local_dimension)

local_dimensions = np.array(local_dimensions)

labels = final_sgd["labels"]
predictions = final_sgd["predictions"]
correct = labels == predictions

local_df = pd.DataFrame({
    "label": labels,
    "prediction": predictions,
    "correct": correct,
    "local_pca_dim90": local_dimensions,
})
local_df.to_csv(CSV_DIR / "local_pca_dimension.csv", index=False)

plt.figure(figsize=(6, 4))
plt.boxplot(
    [local_dimensions[correct], local_dimensions[~correct]],
    tick_labels=["correct", "wrong"],
)
plt.ylabel("Local PCA dimension (90%)")
plt.title("Local geometry: correct vs wrong")
plt.savefig(FIG_DIR / "local_dimension.png", dpi=170)
plt.show()

## 14. SGD와 AdamW의 최종 표현 비교

같은 초기값에서 optimizer만 바꿨을 때 **어느 층부터 representation이 달라지는지** layer별 CKA로 비교합니다.

In [ ]:
final_adamw = adamw_snapshots[EPOCHS]

optimizer_cka_rows = []

for layer in LAYER_NAMES:
    optimizer_cka_rows.append({
        "layer": layer,
        "cka_sgd_vs_adamw": linear_cka(
            final_sgd["features"][layer],
            final_adamw["features"][layer],
        ),
    })

optimizer_cka_df = pd.DataFrame(optimizer_cka_rows)
optimizer_cka_df.to_csv(
    CSV_DIR / "cka_sgd_vs_adamw.csv",
    index=False,
)

plt.figure(figsize=(7, 4))
plt.bar(
    optimizer_cka_df["layer"],
    optimizer_cka_df["cka_sgd_vs_adamw"],
)
plt.ylim(0, 1)
plt.ylabel("Linear CKA")
plt.title("SGD vs AdamW representation CKA")
plt.savefig(FIG_DIR / "cka_sgd_vs_adamw.png", dpi=170)
plt.show()

## 15. Hessian top eigenvalue: loss surface가 얼마나 급하게 휘는가?

Hessian은 loss의 2차 미분 정보입니다. 전체 Hessian은 너무 크므로 만들지 않고, Hessian-vector product와 power iteration으로 **가장 큰 고유값 하나**만 추정합니다.

값이 크면 현재 parameter 주변에 loss가 매우 빠르게 휘는 방향이 있다는 뜻입니다.

In [ ]:
def top_hessian_eigenvalue(state_dict, iterations=8):
    model = SmallCNN().to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    x, y = next(iter(train_eval_loader))
    x = x[:128].to(DEVICE)
    y = y[:128].to(DEVICE)

    parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    vectors = [torch.randn_like(parameter) for parameter in parameters]

    for _ in range(iterations):
        vector_norm = torch.sqrt(
            sum((vector * vector).sum() for vector in vectors)
        ).clamp_min(1e-12)
        vectors = [vector / vector_norm for vector in vectors]

        loss = F.cross_entropy(model(x), y)

        gradients = torch.autograd.grad(
            loss,
            parameters,
            create_graph=True,
        )

        directional_gradient = sum(
            (gradient * vector).sum()
            for gradient, vector in zip(gradients, vectors)
        )

        hessian_vectors = torch.autograd.grad(
            directional_gradient,
            parameters,
        )

        eigenvalue = sum(
            (vector * hessian_vector).sum()
            for vector, hessian_vector in zip(vectors, hessian_vectors)
        ).item()

        vectors = [
            hessian_vector.detach()
            for hessian_vector in hessian_vectors
        ]

    return eigenvalue


hessian_df = pd.DataFrame([
    {
        "condition": "SGD initial",
        "lambda_max": top_hessian_eigenvalue(
            load_checkpoint("sgd", 0)
        ),
    },
    {
        "condition": "SGD final",
        "lambda_max": top_hessian_eigenvalue(
            load_checkpoint("sgd", EPOCHS)
        ),
    },
    {
        "condition": "AdamW final",
        "lambda_max": top_hessian_eigenvalue(
            load_checkpoint("adamw", EPOCHS)
        ),
    },
])

hessian_df.to_csv(CSV_DIR / "hessian_top.csv", index=False)

plt.figure(figsize=(7, 4))
plt.bar(hessian_df["condition"], hessian_df["lambda_max"])
plt.xticks(rotation=15)
plt.ylabel("Estimated top eigenvalue")
plt.title("Hessian top eigenvalue")
plt.savefig(FIG_DIR / "hessian_top.png", dpi=170)
plt.show()

## 16. Weight interpolation: 두 solution 사이에 barrier가 있는가?

두 parameter state를 직선으로 섞습니다.

- `alpha=0` → 모델 A
- `alpha=1` → 모델 B
- 중간 값 → 두 모델 weight의 선형 혼합

중간에서 loss가 크게 솟으면 두 solution이 **직선 low-loss path로 연결되지 않는다**는 뜻입니다.

비교는 `SGD 중간 → SGD final`과 `SGD final → AdamW final` 두 가지입니다.

In [ ]:
def interpolate_state(state_a, state_b, alpha):
    interpolated = {}

    for key in state_a:
        if torch.is_floating_point(state_a[key]):
            interpolated[key] = (
                (1 - alpha) * state_a[key]
                + alpha * state_b[key]
            )
        else:
            interpolated[key] = (
                state_a[key] if alpha < 0.5 else state_b[key]
            )

    return interpolated


@torch.no_grad()
def interpolation_curve(state_a, state_b):
    model = SmallCNN().to(DEVICE)
    rows = []

    for alpha in np.linspace(0, 1, 21):
        model.load_state_dict(
            interpolate_state(state_a, state_b, float(alpha))
        )

        loss, accuracy = evaluate(model, val_loader)

        rows.append({
            "alpha": float(alpha),
            "loss": loss,
            "accuracy": accuracy,
        })

    return pd.DataFrame(rows)


middle_epoch = DIAG_EPOCHS[-2]

same_run_curve = interpolation_curve(
    load_checkpoint("sgd", middle_epoch),
    load_checkpoint("sgd", EPOCHS),
)
same_run_curve["path"] = "SGD middle to SGD final"

cross_optimizer_curve = interpolation_curve(
    load_checkpoint("sgd", EPOCHS),
    load_checkpoint("adamw", EPOCHS),
)
cross_optimizer_curve["path"] = "SGD final to AdamW final"

interpolation_df = pd.concat(
    [same_run_curve, cross_optimizer_curve],
    ignore_index=True,
)
interpolation_df.to_csv(
    CSV_DIR / "weight_interpolation.csv",
    index=False,
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for path_name, part in interpolation_df.groupby("path"):
    axes[0].plot(
        part["alpha"],
        part["loss"],
        marker="o",
        label=path_name,
    )
    axes[1].plot(
        part["alpha"],
        part["accuracy"],
        marker="o",
        label=path_name,
    )

axes[0].set_title("Interpolation loss")
axes[1].set_title("Interpolation accuracy")

for axis in axes:
    axis.set_xlabel("alpha")
    axis.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "weight_interpolation.png", dpi=170)
plt.show()

## 17. TensorBoard와 최종 저장 확인

TensorBoard에서는 batch별 gradient/update 변화와 epoch별 validation metric을 시간축으로 다시 볼 수 있습니다.

마지막 `assert`는 Google Drive 안에 `.pt` 모델 weight가 들어가지 않았는지 검사합니다.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/deep_learning_diagnostics/tensorboard

In [ ]:
summary = {
    "epochs": EPOCHS,
    "diagnostic_epochs": DIAG_EPOCHS,
    "sgd_final_val_accuracy": float(
        sgd_history.iloc[-1]["val_accuracy"]
    ),
    "adamw_final_val_accuracy": float(
        adamw_history.iloc[-1]["val_accuracy"]
    ),
    "linear_probe": dict(
        zip(
            probe_df["layer"],
            probe_df["probe_accuracy"].astype(float),
        )
    ),
    "local_pca_dim90_mean": float(local_dimensions.mean()),
    "hessian": dict(
        zip(
            hessian_df["condition"],
            hessian_df["lambda_max"].astype(float),
        )
    ),
}

with open(SUMMARY_DIR / "experiment_summary.json", "w") as file:
    json.dump(summary, file, indent=2)

assert not any(DRIVE_ROOT.rglob("*.pt")), (
    "Model weight was found on Google Drive."
)

print(json.dumps(summary, indent=2))
print("Storage policy passed: no .pt model weights on Drive.")

## 결과를 읽는 권장 순서

```text
1. loss / accuracy
→ 2. gradient norm + update-to-weight
→ 3. effective rank + linear probe
→ 4. CKA
→ 5. PCA / UMAP
→ 6. local PCA
→ 7. Hessian
→ 8. interpolation
```

예를 들어 `validation 정체 → block2 update가 작음 → block2 effective rank도 거의 안 바뀜 → CKA to init도 계속 높음`이라면 **block2가 feature learning을 거의 하지 못하는가?**라는 가설을 세울 수 있습니다.

그 다음 실험에서는 learning rate, normalization, width, optimizer 중 **하나만 바꿔서** 같은 진단을 다시 돌립니다.